# MLflow Experiment Tracking (Day 7)

## Objective
Add **experiment tracking** to classification models using MLflow for reproducibility and model management.

## Why MLflow?
* **Version Control**: Track every model training run with parameters and metrics
* **Comparison**: Compare runs side-by-side in MLflow UI
* **Model Registry**: Register and version best models for production
* **Reproducibility**: Log everything needed to recreate results

## What We'll Log
* **Parameters**: Model hyperparameters (max_depth, num_trees, regularization)
* **Metrics**: AUC, F1, precision, recall (train & test)
* **Models**: Trained model artifacts
* **Tags**: Metadata (model type, dataset version, user)

## Dataset
* Same as Day 5-6: `h2_gold_model_scoring_base`
* Time-based split (2020 train, 2021 test)
* Target: `high_price_period` (binary classification)

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import mlflow
import mlflow.spark
import pandas as pd
import time

print("✅ Imports loaded (including mlflow)")

In [0]:
# Data configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772775399"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

# Target definition
LABEL_COL = "high_price_period"
TIME_COL = "event_time_utc"  # Fixed: was mtu_cet_cest
TARGET_COL = "price_eur_mwh"
SPLIT_DATE = "2021-01-01"
PRICE_THRESHOLD_PERCENTILE = 0.75

# MLflow configuration
EXPERIMENT_NAME = "/Users/labuser13955035_1772775399@vocareum.com/H2_Price_Classification"

print(f"Source Table: {SOURCE_TABLE}")
print(f"MLflow Experiment: {EXPERIMENT_NAME}")

In [0]:
# Set or create MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

# Get experiment details
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"✅ MLflow Experiment ready")
print(f"  Name: {experiment.name}")
print(f"  ID: {experiment.experiment_id}")
print(f"  Artifact Location: {experiment.artifact_location}")

In [0]:
# Load data
df = spark.table(SOURCE_TABLE)

# Calculate threshold and create label
threshold = df.approxQuantile(TARGET_COL, [PRICE_THRESHOLD_PERCENTILE], 0.01)[0]
df = df.withColumn(
    LABEL_COL,
    F.when(F.col(TARGET_COL) > threshold, 1).otherwise(0)
)

# Time-based split
train = df.filter(F.col(TIME_COL) < F.lit(SPLIT_DATE))
test = df.filter(F.col(TIME_COL) >= F.lit(SPLIT_DATE))

# Feature selection (exclude leakage)
exclude_exact = {LABEL_COL, TIME_COL, TARGET_COL, "area"}
exclude_prefixes = ("next_", "lead_", "future_", "forecast_")
numeric_types = ("double", "float", "int", "bigint", "long", "short", "byte")

feature_cols = []
for field in train.schema.fields:
    name = field.name
    dtype = field.dataType.simpleString().lower()
    if name in exclude_exact:
        continue
    if any(name.startswith(prefix) for prefix in exclude_prefixes):
        continue
    if any(t in dtype for t in numeric_types):
        feature_cols.append(name)

# Calculate class weights
counts = train.groupBy(LABEL_COL).count().collect()
count_map = {int(r[LABEL_COL]): int(r['count']) for r in counts}
n0, n1 = count_map.get(0, 0), count_map.get(1, 0)
total = n0 + n1
w0, w1 = total / (2.0 * n0), total / (2.0 * n1)

train = train.withColumn(
    "class_weight",
    F.when(F.col(LABEL_COL) == 1, F.lit(w1)).otherwise(F.lit(w0))
)
test = test.withColumn(
    "class_weight",
    F.when(F.col(LABEL_COL) == 1, F.lit(w1)).otherwise(F.lit(w0))
)

print(f"Train: {train.count():,} | Test: {test.count():,}")
print(f"Features: {len(feature_cols)}")
print(f"Threshold: {threshold:.2f} EUR/MWh")
print(f"Class weights: w0={w0:.4f}, w1={w1:.4f}")

In [0]:
# Helper function to evaluate models
def evaluate_model(predictions, label_col=LABEL_COL):
    """Evaluate model and return metrics dict"""
    auc_eval = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    
    f1_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="f1"
    )
    
    precision_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="weightedPrecision"
    )
    
    recall_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol="prediction",
        metricName="weightedRecall"
    )
    
    return {
        "auc": auc_eval.evaluate(predictions),
        "f1": f1_eval.evaluate(predictions),
        "precision": precision_eval.evaluate(predictions),
        "recall": recall_eval.evaluate(predictions)
    }

print("✅ Helper functions defined")

In [0]:
print("Training Logistic Regression with MLflow tracking...")
print("="*80)

with mlflow.start_run(run_name="logistic_regression"):
    # Set tags
    mlflow.set_tag("model_type", "logistic_regression")
    mlflow.set_tag("dataset", "h2_gold_model_scoring_base")
    mlflow.set_tag("split_strategy", "time_based_2020_2021")
    
    # Log parameters
    mlflow.log_param("max_iter", 100)
    mlflow.log_param("reg_param", 0.01)
    mlflow.log_param("elastic_net_param", 0.0)
    mlflow.log_param("family", "binomial")
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("class_weight_0", w0)
    mlflow.log_param("class_weight_1", w1)
    
    # Build pipeline
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features",
        handleInvalid="skip"
    )
    
    lr = LogisticRegression(
        featuresCol="features",
        labelCol=LABEL_COL,
        weightCol="class_weight",
        maxIter=100,
        regParam=0.01,
        elasticNetParam=0.0,
        standardization=True,
        family="binomial"
    )
    
    pipeline_lr = Pipeline(stages=[assembler, lr])
    
    # Train
    start_time = time.time()
    model_lr = pipeline_lr.fit(train)
    training_time = time.time() - start_time
    
    # Evaluate on train and test
    pred_train = model_lr.transform(train)
    pred_test = model_lr.transform(test)
    
    train_metrics = evaluate_model(pred_train)
    test_metrics = evaluate_model(pred_test)
    
    # Log metrics
    mlflow.log_metric("train_auc", train_metrics["auc"])
    mlflow.log_metric("train_f1", train_metrics["f1"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall", train_metrics["recall"])
    
    mlflow.log_metric("test_auc", test_metrics["auc"])
    mlflow.log_metric("test_f1", test_metrics["f1"])
    mlflow.log_metric("test_precision", test_metrics["precision"])
    mlflow.log_metric("test_recall", test_metrics["recall"])
    
    mlflow.log_metric("training_time_sec", training_time)
    
    # Log model
    mlflow.spark.log_model(model_lr, "model")
    
    run_id_lr = mlflow.active_run().info.run_id
    
    print(f"✅ Logistic Regression trained")
    print(f"   Test AUC: {test_metrics['auc']:.4f}")
    print(f"   Test F1:  {test_metrics['f1']:.4f}")
    print(f"   Run ID: {run_id_lr}")

In [0]:
print("Training Random Forest with MLflow tracking...")
print("="*80)

with mlflow.start_run(run_name="random_forest"):
    # Set tags
    mlflow.set_tag("model_type", "random_forest")
    mlflow.set_tag("dataset", "h2_gold_model_scoring_base")
    mlflow.set_tag("split_strategy", "time_based_2020_2021")
    
    # Log parameters
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_instances_per_node", 1)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("class_weight_0", w0)
    mlflow.log_param("class_weight_1", w1)
    
    # Build pipeline
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features",
        handleInvalid="skip"
    )
    
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol=LABEL_COL,
        weightCol="class_weight",
        numTrees=100,
        maxDepth=12,
        minInstancesPerNode=1,
        seed=42
    )
    
    pipeline_rf = Pipeline(stages=[assembler, rf])
    
    # Train
    start_time = time.time()
    model_rf = pipeline_rf.fit(train)
    training_time = time.time() - start_time
    
    # Evaluate
    pred_train = model_rf.transform(train)
    pred_test = model_rf.transform(test)
    
    train_metrics = evaluate_model(pred_train)
    test_metrics = evaluate_model(pred_test)
    
    # Log metrics
    mlflow.log_metric("train_auc", train_metrics["auc"])
    mlflow.log_metric("train_f1", train_metrics["f1"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall", train_metrics["recall"])
    
    mlflow.log_metric("test_auc", test_metrics["auc"])
    mlflow.log_metric("test_f1", test_metrics["f1"])
    mlflow.log_metric("test_precision", test_metrics["precision"])
    mlflow.log_metric("test_recall", test_metrics["recall"])
    
    mlflow.log_metric("training_time_sec", training_time)
    
    # Log model
    mlflow.spark.log_model(model_rf, "model")
    
    run_id_rf = mlflow.active_run().info.run_id
    
    print(f"✅ Random Forest trained")
    print(f"   Test AUC: {test_metrics['auc']:.4f}")
    print(f"   Test F1:  {test_metrics['f1']:.4f}")
    print(f"   Run ID: {run_id_rf}")

In [0]:
print("Training Gradient Boosting with MLflow tracking...")
print("="*80)

with mlflow.start_run(run_name="gradient_boosting"):
    # Set tags
    mlflow.set_tag("model_type", "gradient_boosting")
    mlflow.set_tag("dataset", "h2_gold_model_scoring_base")
    mlflow.set_tag("split_strategy", "time_based_2020_2021")
    
    # Log parameters
    mlflow.log_param("max_iter", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("step_size", 0.1)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("class_weight_0", w0)
    mlflow.log_param("class_weight_1", w1)
    
    # Build pipeline
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features",
        handleInvalid="skip"
    )
    
    gbt = GBTClassifier(
        featuresCol="features",
        labelCol=LABEL_COL,
        weightCol="class_weight",
        maxIter=100,
        maxDepth=10,
        stepSize=0.1,
        seed=42
    )
    
    pipeline_gbt = Pipeline(stages=[assembler, gbt])
    
    # Train
    start_time = time.time()
    model_gbt = pipeline_gbt.fit(train)
    training_time = time.time() - start_time
    
    # Evaluate
    pred_train = model_gbt.transform(train)
    pred_test = model_gbt.transform(test)
    
    train_metrics = evaluate_model(pred_train)
    test_metrics = evaluate_model(pred_test)
    
    # Log metrics
    mlflow.log_metric("train_auc", train_metrics["auc"])
    mlflow.log_metric("train_f1", train_metrics["f1"])
    mlflow.log_metric("train_precision", train_metrics["precision"])
    mlflow.log_metric("train_recall", train_metrics["recall"])
    
    mlflow.log_metric("test_auc", test_metrics["auc"])
    mlflow.log_metric("test_f1", test_metrics["f1"])
    mlflow.log_metric("test_precision", test_metrics["precision"])
    mlflow.log_metric("test_recall", test_metrics["recall"])
    
    mlflow.log_metric("training_time_sec", training_time)
    
    # Log model
    mlflow.spark.log_model(model_gbt, "model")
    
    run_id_gbt = mlflow.active_run().info.run_id
    
    print(f"✅ Gradient Boosting trained")
    print(f"   Test AUC: {test_metrics['auc']:.4f}")
    print(f"   Test F1:  {test_metrics['f1']:.4f}")
    print(f"   Run ID: {run_id_gbt}")

In [0]:
# Get all runs from the experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Display runs sorted by test_auc
runs_display = runs[[
    "run_id",
    "tags.model_type",
    "metrics.test_auc",
    "metrics.test_f1",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.training_time_sec"
]].sort_values("metrics.test_auc", ascending=False)

print("="*80)
print("MLFLOW EXPERIMENT RUNS COMPARISON")
print("="*80)
display(runs_display)

# Identify best run
best_run = runs_display.iloc[0]
print(f"\n🏆 BEST RUN")
print(f"   Model: {best_run['tags.model_type']}")
print(f"   Test AUC: {best_run['metrics.test_auc']:.4f}")
print(f"   Test F1:  {best_run['metrics.test_f1']:.4f}")
print(f"   Run ID: {best_run['run_id']}")
print(f"\n✅ All runs logged to MLflow")

## 📊 Viewing Results in MLflow UI

### Access MLflow UI
1. Click **Experiments** icon in left sidebar
2. Find experiment: `H2_Price_Classification`
3. View all runs with metrics and parameters

### Compare Runs
1. Select multiple runs using checkboxes
2. Click **Compare** button
3. View side-by-side:
   * Parameters (max_depth, num_trees, etc.)
   * Metrics (AUC, F1, precision, recall)
   * Charts (metric trends)

### Key Features
* **Parallel Coordinates**: Visualize parameter vs metric relationships
* **Scatter Plots**: Compare two metrics across runs
* **Run Details**: Click any run to see:
  * Full parameters
  * All metrics (train + test)
  * Model artifacts
  * Tags and metadata

### Load a Model
```python
# Load best model
run_id = "<your_best_run_id>"
model = mlflow.spark.load_model(f"runs:/{run_id}/model")

# Make predictions
predictions = model.transform(test)
```

## Next Steps
* **Day 8-9**: Build production inference pipeline
* **Model Registry**: Register best model for production deployment